# Advisor Audit Environment Prompt Demo

This notebook walks through one manual month of the advisor-audit environment and shows the exact prompts at each step.

It focuses on:
- the advisor recommendation prompt
- the investor decision prompt
- the next-month advisor prompt after resolution


In [ ]:
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace
import importlib
import sys

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = next((candidate for candidate in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents] if (candidate / 'Environments').exists() and (candidate / 'LocalizationScripts').exists()), NOTEBOOK_ROOT)
ENV_SRC = REPO_ROOT / 'Environments' / 'AdvisorAudit' / 'src'
if str(ENV_SRC) not in sys.path:
    sys.path.insert(0, str(ENV_SRC))

import financial_advisor_environment as advisor_env
importlib.reload(advisor_env)

FinancialAdvisorAuditEnvironment = advisor_env.FinancialAdvisorAuditEnvironment
AdvisorAuditSpec = advisor_env.AdvisorAuditSpec
DEFAULT_SCENARIOS = advisor_env.DEFAULT_SCENARIOS
print('Imported from:', ENV_SRC / 'financial_advisor_environment.py')


In [ ]:
def make_agents():
    return [
        SimpleNamespace(name='Advisor', reasoning_instruction='COD', instruction_format='default'),
        SimpleNamespace(name='Investor', reasoning_instruction='COD', instruction_format='default'),
    ]


def make_env(seed=0, scenario_name='retirement_conservative'):
    spec = AdvisorAuditSpec()
    return FinancialAdvisorAuditEnvironment(
        agents=make_agents(),
        seed=seed,
        spec=spec,
        scenario_name=scenario_name,
        history_window=10,
    )


def show_messages(messages):
    for idx, msg in enumerate(messages):
        print(f'===== MESSAGE {idx} / role={msg["role"]} =====')
        print(msg['content'])
        print()


def show_state(env, include_system_prompt=True):
    state = env.get_state(include_system_prompt=include_system_prompt)
    print('phase:', state['phase'])
    print('active_player:', state['active_player'])
    print('month:', state['month_name'], f"({state['month_index'] + 1}/{state['max_months']})")
    print('portfolio_value:', state['portfolio_value'])
    print('advisor_reputation:', state['advisor_reputation'])
    print('outcome:', state['outcome'])
    print('recent_history:', env.history[-5:])
    print()
    show_messages(state['messages'])
    return state


In [ ]:
env = make_env(seed=0, scenario_name='retirement_conservative')
_ = show_state(env, include_system_prompt=True)


In [ ]:
truthful_advisor_action = env.get_truthful_action()
advisor_result = env.manual_step(truthful_advisor_action)

print('truthful advisor action:')
pprint(truthful_advisor_action)
print('\nadvisor result:')
pprint(advisor_result)
print('\nnext prompt (investor decision):')
_ = show_state(env, include_system_prompt=True)


In [ ]:
truthful_investor_action = env.get_truthful_action()
investor_result = env.manual_step(truthful_investor_action)

print('truthful investor action:')
pprint(truthful_investor_action)
print('\ninvestor result:')
pprint(investor_result)
print('\nnext prompt (next month advisor phase):')
_ = show_state(env, include_system_prompt=True)
